# Integrative analysis of drug-tolerant persister (DTP) states in HER2-positive breast cancer — master analysis notebook

This notebook consolidates the full computational analysis for the manuscript into one sectioned pipeline.
Each section reproduces one part of the study and its figures.

| Section | Analysis | Figure | Primary data |
|---|---|---|---|
| 1 | scRNA-seq DTP discovery and signature derivation | Fig 1 | GEO **GSE156246** |
| 2 | Pathway enrichment (GSEA) and module scoring | Figs 2–4 | derived from Section 1 |
| 3 | Clinical survival analysis + proportional-hazards test | Fig 5 | cBioPortal **METABRIC**, **TCGA** |
| 4 | Perturbational (LINCS/iLINCS) drug repurposing | Fig 6 | **iLINCS**, Enrichr, ChEMBL |
| 5 | Spatial transcriptomics + patient-level sensitivity | Fig 7 | **Zenodo 4751624** (Andersson et al.) |

**Data accessions**
- Single-cell RNA-seq: GEO `GSE156246` (Chang et al., 2022)
- Spatial transcriptomics: Zenodo `4751624` / GitHub `almaan/her2st` (Andersson et al., 2021)
- Clinical cohorts: cBioPortal `brca_metabric`; TCGA breast cohort (curatedTCGAData)
- Perturbational signatures: iLINCS platform (`findConcordancesSC`)

Sections 3 and 5 are self-contained (they fetch their own data). Sections 1, 2 and 4 share the single-cell object and signatures produced in Section 1.

## Section 0 — Environment setup

In [ ]:
!pip install -q scanpy anndata gseapy scipy statsmodels scikit-learn lifelines squidpy rpy2 requests networkx openpyxl chembl_webresource_client

import os, re, json, warnings, zipfile, urllib.request, tarfile, shutil
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import sparse, stats
from scipy.stats import rankdata, mannwhitneyu, norm
from statsmodels.stats.multitest import multipletests

SEED = 42
np.random.seed(SEED); sc.settings.seed = SEED; sc.settings.verbosity = 1
for d in ["outputs/figures","outputs/signatures","outputs/tables"]:
    Path(d).mkdir(parents=True, exist_ok=True)

def rank_biserial(U, n1, n2):  # effect size for Mann-Whitney
    return 1 - (2*U)/(n1*n2)

print("Environment ready:", sc.__version__)

## Section 1 — scRNA-seq DTP discovery and signatures (Figure 1)

Reanalysis of GSE156246 (BT474 and HCC1419, each with untreated / acute-treatment / persister conditions).
Pipeline: load 10x matrices -> QC -> normalize/log1p -> HVG -> Leiden clustering ->
DTP-enriched clusters (>= 60% persister-condition cells) -> Wilcoxon DE (log2FC>0.25, BH padj<0.05) ->
shared (intersection) and lineage-specific (set difference) DTP signatures.

In [ ]:
# --- Download GSE156246 (10x matrices for 6 samples) ---
GEO_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE156nnn/GSE156246/suppl/GSE156246_RAW.tar"
BASE = Path("data/GSE156246")
SAMPLES = {
    "BT474_DTP":("GSM4727545_BT474_DTP","BT474","DTP"),
    "BT474_Treatment":("GSM4727546_BT474_Treatment","BT474","Treatment"),
    "BT474_Untreated":("GSM4727550_BT474_Untreated","BT474","Untreated"),
    "HCC1419_DTP":("GSM4727547_HCC1419-DTP","HCC1419","DTP"),
    "HCC1419_Treatment":("GSM4727548_HCC1419_Treatment","HCC1419","Treatment"),
    "HCC1419_Untreated":("GSM4727549_HCC1419_Untreated","HCC1419","Untreated"),
}
if not BASE.exists():
    BASE.mkdir(parents=True)
    urllib.request.urlretrieve(GEO_URL, "GSE156246_RAW.tar")
    with tarfile.open("GSE156246_RAW.tar") as t: t.extractall(BASE)
    os.remove("GSE156246_RAW.tar")

def organise(prefix, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    for src in BASE.glob(f"{prefix}_*.gz"):
        stem = src.name.split("_",2)[-1].replace("genes","features")
        if not (dest/stem).exists(): shutil.copy(src, dest/stem)
for name,(prefix,_,_) in SAMPLES.items(): organise(prefix, BASE/name)

In [ ]:
from scipy.io import mmread
def load_10x(folder, sample):
    mtx = mmread(str(Path(folder)/ [f for f in os.listdir(folder) if f.endswith('matrix.mtx.gz')][0])).T.tocsr()
    bc  = pd.read_csv(Path(folder)/[f for f in os.listdir(folder) if 'barcodes' in f][0], header=None)[0].astype(str).tolist()
    ft  = pd.read_csv(Path(folder)/[f for f in os.listdir(folder) if 'features' in f][0], header=None, sep='\t')
    genes = (ft.iloc[:,1] if ft.shape[1]>1 else ft.iloc[:,0]).astype(str).tolist()
    ad = sc.AnnData(X=mtx); ad.obs_names=bc; ad.var_names=genes; ad.obs['sample']=sample
    return ad

ads=[]
for name,(prefix,line,cond) in SAMPLES.items():
    ad = load_10x(BASE/name, name); ad.obs['cell_line']=line; ad.obs['condition']=cond; ads.append(ad)
adata = sc.concat(ads, index_unique=None); adata.obs_names_make_unique()

sc.pp.filter_cells(adata, min_genes=200); sc.pp.filter_genes(adata, min_cells=3)
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
adata = adata[adata.obs['pct_counts_mt']<10].copy()

sc.pp.normalize_total(adata, target_sum=1e4); sc.pp.log1p(adata)
adata.raw = adata.copy()
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var['highly_variable']]
sc.pp.scale(adata, max_value=10); sc.tl.pca(adata, n_comps=50)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=30); sc.tl.umap(adata); sc.tl.leiden(adata, resolution=0.5)
print("Leiden clusters:", adata.obs['leiden'].nunique())

In [ ]:
# --- DTP-enriched clusters at 60% persister threshold ---
adata.obs['is_DTP'] = (adata.obs['condition']=='DTP').astype(int)
THR = 0.6
stats_df = (adata.obs.groupby(['cell_line','leiden'], observed=True)
            .agg(n_cells=('is_DTP','size'), n_DTP=('is_DTP','sum')).reset_index())
stats_df['frac_DTP'] = stats_df['n_DTP']/stats_df['n_cells']
dtp_clusters = (stats_df.query('frac_DTP >= @THR').groupby('cell_line', observed=True)['leiden'].apply(list).to_dict())
print("DTP-enriched clusters:", dtp_clusters)
stats_df.to_csv("outputs/tables/SuppTable_S4_cluster_DTP_fractions.csv", index=False)

adata.obs['DTP_state'] = 'non_DTP'
for line, cl in dtp_clusters.items():
    m = (adata.obs['cell_line']==line) & (adata.obs['leiden'].isin(cl))
    adata.obs.loc[m,'DTP_state'] = f'{line}_DTP'

def clean_gene(g):
    return str(g).split('\t')[1].strip() if '\t' in str(g) else str(g).strip()

def deg_up(line):
    sub = adata[adata.obs['cell_line']==line].copy()
    sc.tl.rank_genes_groups(sub, 'DTP_state', groups=[f'{line}_DTP'], reference='non_DTP', method='wilcoxon')
    df = sc.get.rank_genes_groups_df(sub, group=f'{line}_DTP'); df['names']=df['names'].apply(clean_gene)
    return df[(df['logfoldchanges']>0.25)&(df['pvals_adj']<0.05)]

bt_up, hcc_up = deg_up('BT474'), deg_up('HCC1419')
bt, hcc = set(bt_up['names']), set(hcc_up['names'])
shared, bt_spec, hcc_spec = sorted(bt&hcc), sorted(bt-hcc), sorted(hcc-bt)
print(f"Shared={len(shared)}  BT474-specific={len(bt_spec)}  HCC1419-specific={len(hcc_spec)}")
for nm,gl in [("shared",shared),("bt474",bt_spec),("hcc1419",hcc_spec)]:
    pd.DataFrame({'gene':gl}).to_csv(f"outputs/signatures/{nm}_list.csv", index=False)
bt_up.to_csv("outputs/tables/BT474_DTP_DE_results.csv", index=False)
hcc_up.to_csv("outputs/tables/HCC1419_DTP_DE_results.csv", index=False)

## Section 2 — Pathway enrichment and module scoring (Figures 2–4)

Uses the single-cell object from Section 1. GSEA is run per lineage on a t-statistic pre-ranked list
(MSigDB Hallmark, 1,000 permutations, seed 42). Module scores (proliferation / E2F, luminal, stress-hypoxia,
EMT, and DTP marker genes) are compared between DTP and non-DTP cells with Mann-Whitney U tests and
rank-biserial effect sizes.

In [ ]:
import gseapy as gp
# Gene-set module scores (scored on full gene space via adata.raw)
HALLMARK = gp.get_library('MSigDB_Hallmark_2020', organism='Human')
PROGRAMS = {
    'E2F_targets': HALLMARK['E2F Targets'], 'G2M_checkpoint': HALLMARK['G2-M Checkpoint'],
    'MYC_targets': HALLMARK['Myc Targets V1'], 'Hypoxia': HALLMARK['Hypoxia'],
    'Glycolysis': HALLMARK['Glycolysis'], 'EMT': HALLMARK['Epithelial Mesenchymal Transition'],
    'Luminal': ['FOXA1','GATA3','ESR1','KRT8','KRT18','KRT19','XBP1','SPDEF','AR','ERBB3'],
    'Basal':   ['KRT5','KRT14','KRT17','TP63','EGFR','CDH3','LAMC2','ITGA6','VIM','FN1'],
}
adata_full = adata.raw.to_adata()
for name, genes in PROGRAMS.items():
    gi = [g for g in genes if g in adata_full.var_names]
    if gi: sc.tl.score_genes(adata_full, gi, score_name=f'score_{name}')
adata.obs = adata.obs.join(adata_full.obs[[c for c in adata_full.obs.columns if c.startswith('score_')]])

In [ ]:
# Module-score comparison DTP vs non-DTP (per lineage), with rank-biserial effect size
rows=[]
for line in ['BT474','HCC1419']:
    for prog in [c for c in adata.obs.columns if c.startswith('score_')]:
        d = adata.obs[adata.obs['cell_line']==line]
        a = d[d['DTP_state']==f'{line}_DTP'][prog].dropna()
        b = d[d['DTP_state']=='non_DTP'][prog].dropna()
        if len(a)<10 or len(b)<10: continue
        U,p = mannwhitneyu(a,b, alternative='two-sided')
        rows.append(dict(cell_line=line, program=prog, U=U, p=p,
                         effect_r=rank_biserial(U,len(a),len(b)), n_DTP=len(a), n_non=len(b)))
mod = pd.DataFrame(rows); mod['p_adj']=multipletests(mod['p'], method='fdr_bh')[1]
mod.to_csv("outputs/tables/module_score_comparisons.csv", index=False)
mod.head(12)

In [ ]:
# GSEA (Hallmark, pre-ranked by t-statistic) per lineage -> Figure 2A
from scipy.stats import ttest_ind
def prerank_gsea(line):
    sub = adata.raw.to_adata()[adata.obs['cell_line']==line].copy()
    state = adata.obs.loc[sub.obs_names,'DTP_state']
    A = sub[state==f'{line}_DTP'].X; B = sub[state=='non_DTP'].X
    A = A.toarray() if hasattr(A,'toarray') else A; B = B.toarray() if hasattr(B,'toarray') else B
    t,_ = ttest_ind(A, B, axis=0, equal_var=False)
    rnk = pd.DataFrame({0:sub.var_names, 1:np.nan_to_num(t)}).sort_values(1, ascending=False)
    res = gp.prerank(rnk=rnk, gene_sets='MSigDB_Hallmark_2020', permutation_num=1000,
                     seed=42, min_size=5, max_size=1000, outdir=None)
    return res.res2d
gsea_bt  = prerank_gsea('BT474')
gsea_hcc = prerank_gsea('HCC1419')
gsea_bt.to_csv("outputs/tables/GSEA_BT474.csv", index=False)
gsea_hcc.to_csv("outputs/tables/GSEA_HCC1419.csv", index=False)
gsea_bt.head(10)

## Section 3 — Clinical survival analysis + proportional-hazards test (Figure 5)

METABRIC (HER2-enriched by `CLAUDIN_SUBTYPE = Her2`, n=224) and TCGA (PAM50 HER2-enriched) are fetched
from cBioPortal. Signature scores are mean z-scored expression across matched genes. Overall survival is
modelled with univariate (High-vs-Low tertile) and multivariate (adjusted for age and positive lymph nodes)
Cox regression (lifelines). The proportional-hazards assumption is tested with the Schoenfeld residual test.

*This section is self-contained — upload `shared_list_day2.csv`, `bt_list_day2.csv`, `hcc_list_day2.csv`
(or use the Section 1 outputs in `outputs/signatures/`).*

In [ ]:
import requests
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import proportional_hazard_test, multivariate_logrank_test

BASE="https://www.cbioportal.org/api"; HEAD={"accept":"application/json","Content-Type":"application/json"}
def _load_sig(name, fallback):
    for p in [f"{name}", f"outputs/signatures/{fallback}"]:
        if os.path.exists(p): return pd.read_csv(p)["gene"].tolist()
    raise FileNotFoundError(name)
shared_sig=_load_sig("shared_list_day2.csv","shared_list.csv")
bt_sig    =_load_sig("bt_list_day2.csv","bt474_list.csv")
hcc_sig   =_load_sig("hcc_list_day2.csv","hcc1419_list.csv")
all_genes=list(set(shared_sig+bt_sig+hcc_sig))

clin = pd.DataFrame(requests.get(f"{BASE}/studies/brca_metabric/clinical-data",
        params={"clinicalDataType":"PATIENT","projection":"SUMMARY"}, headers=HEAD).json())
cw = clin.pivot(index="patientId", columns="clinicalAttributeId", values="value").reset_index()
her2 = cw[cw["CLAUDIN_SUBTYPE"]=="Her2"].copy()
samp = pd.DataFrame(requests.get(f"{BASE}/studies/brca_metabric/samples", params={"projection":"SUMMARY"}, headers=HEAD).json())
sids = samp[samp["patientId"].isin(her2["patientId"])]["sampleId"].tolist()
gdf = pd.DataFrame(requests.post(f"{BASE}/genes/fetch", json=all_genes, params={"geneIdType":"HUGO_GENE_SYMBOL"}, headers=HEAD).json())
ez = gdf["entrezGeneId"].astype(str).tolist(); ez2sym = dict(zip(gdf["entrezGeneId"].astype(str), gdf["hugoGeneSymbol"].str.upper()))
expr=[]
for i in range(0,len(sids),100):
    r=requests.post(f"{BASE}/molecular-profiles/brca_metabric_mrna_median_all_sample_Zscores/molecular-data/fetch",
                    json={"entrezGeneIds":ez,"sampleIds":sids[i:i+100]}, headers=HEAD)
    if r.status_code==200 and len(r.text)>2: expr.append(pd.DataFrame(r.json()))
E=pd.concat(expr, ignore_index=True); E["geneSymbol"]=E["entrezGeneId"].astype(str).map(ez2sym)
def score(g): s=E[E["geneSymbol"].isin([x.upper() for x in g])]; return s.groupby("sampleId")["value"].mean()
sc_df=pd.DataFrame({"sampleId":score(shared_sig).index,"Shared_DTP":score(shared_sig).values,
     "BT474_DTP":score(bt_sig).reindex(score(shared_sig).index).values,
     "HCC1419_DTP":score(hcc_sig).reindex(score(shared_sig).index).values})
cs = her2.merge(samp[["patientId","sampleId"]], on="patientId").merge(sc_df, on="sampleId")
cs["OS_years"]=pd.to_numeric(cs["OS_MONTHS"],errors="coerce")/12
cs["OS_event"]=(cs["OS_STATUS"]=="1:DECEASED").astype(int)
cs["age"]=pd.to_numeric(cs["AGE_AT_DIAGNOSIS"],errors="coerce")
cs["lymph"]=pd.to_numeric(cs["LYMPH_NODES_EXAMINED_POSITIVE"],errors="coerce")
cs=cs.dropna(subset=["OS_years","HCC1419_DTP"])
tert=lambda s:(pd.cut(s,[-np.inf,*s.quantile([1/3,2/3]),np.inf],labels=["Low","Mid","High"]))
for k in ["HCC1419_DTP","Shared_DTP","BT474_DTP"]: cs[k.replace("_DTP","_tertile")]=tert(cs[k])
print(f"METABRIC HER2-enriched: n={len(cs)}, events={int(cs['OS_event'].sum())}")

In [ ]:
# Cox models (univariate tertile + multivariate age+lymph) and Schoenfeld PH test
for score_col,tcol in [("HCC1419_DTP","HCC1419_tertile"),("Shared_DTP","Shared_tertile"),("BT474_DTP","BT474_tertile")]:
    sub=cs[cs[tcol].isin(["Low","High"])].copy(); sub["High"]=(sub[tcol]=="High").astype(int)
    d1=sub.dropna(subset=["OS_years","OS_event","High"])
    uni=CoxPHFitter().fit(d1[["OS_years","OS_event","High"]],"OS_years","OS_event")
    d2=sub.dropna(subset=["OS_years","OS_event","High","age","lymph"])
    adj=CoxPHFitter().fit(d2[["OS_years","OS_event","High","age","lymph"]],"OS_years","OS_event")
    hr=lambda m:(np.exp(m.params_["High"]),np.exp(m.confidence_intervals_.loc["High","95% lower-bound"]),np.exp(m.confidence_intervals_.loc["High","95% upper-bound"]),m.summary.loc["High","p"])
    print(f"\n{score_col}: univariate HR={hr(uni)[0]:.2f} ({hr(uni)[1]:.2f}-{hr(uni)[2]:.2f}) p={hr(uni)[3]:.3f} | "
          f"adjusted HR={hr(adj)[0]:.2f} ({hr(adj)[1]:.2f}-{hr(adj)[2]:.2f}) p={hr(adj)[3]:.3f}")
    proportional_hazard_test(adj, d2[["OS_years","OS_event","High","age","lymph"]], time_transform="rank").print_summary(decimals=3)

## Section 4 — Perturbational drug repurposing (LINCS/iLINCS) (Figure 6)

DTP signatures are submitted to iLINCS `findConcordancesSC` to retrieve concordant perturbagens. Compound-level
scores aggregate signatures at the drug level, retaining the **maximum concordance z-score across experimental
conditions**. Candidates are prioritised by (i) overlap of their protein targets with a DTP GO-axis gene universe
(Enrichr GO Biological Process, union of the top-100 enriched terms per signature; >=1 overlapping target),
(ii) cross-lineage concordance (present in both BT474- and HCC1419-derived programs), and (iii) a nonredundant
shortlist (highest-ranked compound per axis target gene). Compounds are annotated with ChEMBL clinical phase.

*No fixed concordance z-score cut-off is applied; all returned perturbagens are retained and then prioritised.*

In [ ]:
import requests
ILINCS = "https://www.ilincs.org/api/ilincsR/findConcordancesSC"
def ilincs_concordances(gene_list, timeout=180):
    up = [g for g in gene_list]
    payload = {"signatureProfile": {"genes": up, "values": [1.0]*len(up)}, "library": "LINCS"}
    try:
        r = requests.post(ILINCS, json=payload, timeout=timeout)
        data = r.json()
        for k in ["data","results","concordances","table"]:
            if isinstance(data, dict) and k in data: data = data[k]; break
        x = pd.DataFrame(data)
    except Exception as e:
        print("iLINCS fetch note:", e); return pd.DataFrame(columns=["compound","zScore","ProteinTargets"])
    for c in ["zScores","z","zscore","score","connectivityScore","maxZ"]:
        if c in x.columns: x = x.rename(columns={c:"zScore"})
    x["zScore"] = pd.to_numeric(x.get("zScore"), errors="coerce"); x = x.dropna(subset=["zScore"])
    keep=[c for c in ["compound","zScore","ProteinTargets"] if c in x.columns]
    return x[keep].copy()

raw_bt  = ilincs_concordances(bt_sig)
raw_hcc = ilincs_concordances(hcc_sig)

In [ ]:
# Aggregate to drug level (max concordance z across conditions), merge, cross-lineage
def summarize(df, name):
    if df.empty: return pd.DataFrame(columns=["compound","Max_z","ProteinTargets","Signature"])
    first=lambda s:(s[s.astype(str).str.strip()!=""].iloc[0] if (s.astype(str).str.strip()!="").any() else "")
    out=(df.groupby("compound",as_index=False)
           .agg(Max_z=("zScore","max"), N_conditions=("zScore","count"),
                ProteinTargets=("ProteinTargets", first)))
    out["Signature"]=name; return out.sort_values("Max_z",ascending=False)
bt_drug, hcc_drug = summarize(raw_bt,"BT474_DTP"), summarize(raw_hcc,"HCC1419_DTP")
merged = (bt_drug.rename(columns={"Max_z":"BT474_z","ProteinTargets":"Targets_BT"})[["compound","BT474_z","Targets_BT"]]
          .merge(hcc_drug.rename(columns={"Max_z":"HCC1419_z","ProteinTargets":"Targets_HCC"})[["compound","HCC1419_z","Targets_HCC"]],
                 on="compound", how="outer"))
merged["Cross_Signature"]=merged["BT474_z"].notna() & merged["HCC1419_z"].notna()
merged["Max_z_overall"]=merged[["BT474_z","HCC1419_z"]].max(axis=1)
merged["Targets_All"]=merged.apply(lambda r:"|".join(dict.fromkeys([t for t in [str(r.get("Targets_BT","")),str(r.get("Targets_HCC",""))] if t and t.lower()!="nan"])), axis=1)
print(f"BT474 drugs={len(bt_drug)}, HCC1419 drugs={len(hcc_drug)}, merged unique={len(merged)}")

In [ ]:
# DTP GO-axis gene universe (Enrichr GO_BP, top-100 terms per signature) + target overlap + shortlist
import gseapy as gp
def go_axis(gene_list, top_n=100):
    try:
        enr = gp.enrichr(gene_list=list(gene_list), gene_sets="GO_Biological_Process_2021", outdir=None).results
    except Exception as e:
        print("Enrichr note:", e); return set()
    enr = enr.sort_values("Adjusted P-value").head(top_n)
    genes=set()
    for s in enr["Genes"]: genes |= set(str(s).upper().split(";"))
    return {g.strip() for g in genes if g.strip()}
axis_union = go_axis(bt_sig) | go_axis(hcc_sig) | go_axis(shared_sig)
print("DTP-axis gene universe:", len(axis_union))

def parse_targets(s): return [t.strip().upper() for t in re.split(r"[|;,/]+", str(s)) if t.strip()]
merged["Axis_overlap_genes"]=merged["Targets_All"].apply(lambda s:[g for g in parse_targets(s) if g in axis_union])
merged["Axis_overlap_count"]=merged["Axis_overlap_genes"].apply(len)
filtered = merged[merged["Axis_overlap_count"]>0].copy()
cross = filtered[filtered["Cross_Signature"]].copy().sort_values("Max_z_overall", ascending=False)
# nonredundant shortlist: highest-ranked compound per axis target gene
ex = cross.explode("Axis_overlap_genes")
top_per = ex.sort_values("Max_z_overall", ascending=False).drop_duplicates("Axis_overlap_genes")
shortlist = top_per.drop_duplicates("compound").sort_values("Max_z_overall", ascending=False)
print(f"axis-overlapping={len(filtered)}, cross-lineage={len(cross)}, shortlist={len(shortlist)}")
shortlist[["compound","Axis_overlap_genes","Max_z_overall"]].to_csv("outputs/tables/SuppTable_perturbagen_shortlist.csv", index=False)
shortlist.head(15)

In [ ]:
# ChEMBL clinical-phase annotation of the shortlist
try:
    from chembl_webresource_client.new_client import new_client
    mol = new_client.molecule
    def max_phase(name):
        try:
            hits=mol.filter(pref_name__iexact=name).only(["max_phase"])
            return hits[0]["max_phase"] if hits else None
        except Exception: return None
    shortlist["ChEMBL_max_phase"]=shortlist["compound"].apply(max_phase)
except Exception as e:
    print("ChEMBL note:", e)
shortlist[["compound","Max_z_overall","ChEMBL_max_phase"]] if "ChEMBL_max_phase" in shortlist else shortlist.head()

## Section 5 — Spatial transcriptomics + patient-level sensitivity (Figure 7)

Spatial data (Andersson et al., 2021; GitHub `almaan/her2st`, 36 sections / 8 patients) are loaded, deconvolution
major-cell-type proportions attached, and the shared DTP signature scored per spot (rank-based). DTP activity is
residualised on epithelial content and sequencing depth. Per-section regressions link each microenvironment program
to DTP activity; section estimates are combined by inverse-variance fixed-effect meta-analysis (BH-corrected).
A **patient-level sensitivity analysis** (section estimates combined within each of the 8 patients, then meta-analyzed)
and a **random-effects model with I²** address non-independence of sections from the same patient.

*This section is self-contained — it clones the spatial dataset and uses `shared_list_day2.csv`.*

In [ ]:
import numpy as np, pandas as pd, re, zipfile
from glob import glob
from pathlib import Path
import statsmodels.api as sm
from scipy.stats import rankdata, norm

REPO=Path("her2st_repo")
if not REPO.exists():
    import subprocess; subprocess.run(["git","clone","--depth","1","https://github.com/almaan/her2st.git",str(REPO)]); CNT=REPO/"data"/"ST-cnts"; SPOT=REPO/"data"/"ST-spotfiles"
DECONV=REPO/"res"/"ST-deconv"/"props"/"major.zip"

def sid_of(p):
    m=re.match(r"^([A-H]\d+)", Path(p).name); return m.group(1) if m else None

def load_section(sid):
    dc=pd.read_csv(CNT/f"{sid}.tsv.gz", sep="\t")
    ds=pd.read_csv(SPOT/f"{sid}_selection.tsv", sep="\t")
    spot_ids=dc.iloc[:,0].astype(str)
    xy=spot_ids.str.extract(r"^(?P<x>\d+)x(?P<y>\d+)$").astype(int)
    dcxy=pd.DataFrame({"spot_id":spot_ids.values,"x":xy["x"].values,"y":xy["y"].values})
    ds["x"]=ds["x"].astype(int); ds["y"]=ds["y"].astype(int)
    merged=dcxy.merge(ds[["x","y"]],on=["x","y"],how="inner")
    X=dc.iloc[:,1:].values.astype(float)
    tc=X.sum(1); ng=(X>0).sum(1)
    mc=np.median(tc); MADc=np.median(np.abs(tc-mc)); mg=np.median(ng); MADg=np.median(np.abs(ng-mg))
    keep=(tc>=max(200,mc-3*MADc))&(ng>=max(200,mg-3*MADg))
    X=X[keep]; merged=merged.loc[keep].reset_index(drop=True); tc=tc[keep]
    gd=(X>0).sum(0); keep_g=gd>=max(5,int(0.01*X.shape[0]))
    X=X[:,keep_g]; genes=np.array(dc.columns[1:])[keep_g]
    # normalize_total 1e4 + log1p
    Xn=np.log1p(X/ X.sum(1,keepdims=True) *1e4)
    obs=pd.DataFrame({"spot_id":merged["spot_id"].values,"total_counts":tc})
    obs=obs.set_index("spot_id")
    return {"X":Xn,"genes":{g:i for i,g in enumerate(genes)},"var":list(genes),"obs":obs}

sections=sorted([sid_of(p) for p in glob(str(CNT/"*.tsv.gz"))])
D={s:load_section(s) for s in sections}
print("Loaded sections:",len(D))

# deconv epithelial
def epi(sid, idx):
    with zipfile.ZipFile(DECONV) as z:
        df=pd.read_csv(z.open(f"major/{sid}-proportion.tsv"),sep="\t").rename(columns={"Unnamed: 0":"spot_id"})
    df["spot_id"]=df["spot_id"].astype(str); df=df.set_index("spot_id")
    return df.loc[idx,"Epithelial"].values
for s in sections:
    D[s]["obs"]["epi"]=epi(s, D[s]["obs"].index)

shared=[g.upper() for g in pd.read_csv("shared_list_day2.csv")["gene"]]
PROG={"IFN_GAMMA":["STAT1","IRF1","CXCL9","CXCL10","CXCL11","GBP1","GBP2","IFI44L","ISG15","HLA-DRA"],
 "HYPOXIA":["CA9","VEGFA","SLC2A1","LDHA","ALDOA","ENO1","PGK1","P4HA1","ADM","BNIP3"],
 "ECM_CAF":["COL1A1","COL1A2","COL3A1","COL5A1","FN1","SPARC","DCN","LUM","ACTA2","TAGLN"],
 "GLYCOLYSIS":["SLC2A1","HK2","PFKP","ALDOA","GAPDH","PGK1","ENO1","PKM","LDHA","PFKM"]}

def dtp_rank(d):  # cell 36 scorer, shared sig
    X=d["X"]; idx=[d["genes"][g] for g in shared if g in d["genes"]]
    ranks=np.argsort(np.argsort(-X,axis=1),axis=1); tot=X.shape[1]
    r=((tot-ranks[:,idx]).mean(1)/tot); return (r-r.mean())/(r.std() or 1)

def prog_rank(d,gs):  # cell 75 full-universe scorer
    X=d["X"]; present=[g for g in gs if g in d["genes"]]
    ra=np.apply_along_axis(rankdata,1,X,method="average")
    idx=[d["genes"][g] for g in present]
    v=ra[:,idx].mean(1); return (v-v.mean())/(v.std() or 1)

rows=[]
def zz(v):
    v=np.asarray(v,float); sd=v.std()
    return (v-v.mean())/(sd if sd>0 else 1)
for s in sections:
    d=D[s]; o=d["obs"]
    o["DTP_z"]=dtp_rank(d)
    Xc=o[["epi","total_counts"]].astype(float); Xc=(Xc-Xc.mean())/Xc.std(); Xc=sm.add_constant(Xc)
    o["DTP_resid"]=sm.OLS(o["DTP_z"].astype(float),Xc).fit().resid
    yv=zz(o["DTP_resid"].values)   # cell 91: y = zscore(DTP_SHARED_resid)
    for k,gs in PROG.items():
        xv=zz(prog_rank(d,gs))     # cell 91: x = zscore(program)
        mask=np.isfinite(yv)&np.isfinite(xv)
        if mask.sum()<50: continue
        Xo=sm.add_constant(xv[mask]); m=sm.OLS(yv[mask],Xo).fit()
        rows.append(dict(section=s,patient=s[0],program=k,
            beta=float(m.params[1]),se=float(m.bse[1]),n=int(mask.sum())))
assoc=pd.DataFrame(rows); assoc.to_csv("outputs/tables/spatial_per_section.csv",index=False)

def meta(b,sd):
    b=np.asarray(b,float); sd=np.asarray(sd,float); k=len(b); df=k-1
    w=1/sd**2; bfe=np.sum(w*b)/np.sum(w); sefe=np.sqrt(1/np.sum(w))
    Q=float(np.sum(w*(b-bfe)**2)); I2=max(0,(Q-df)/Q)*100 if Q>0 and df>0 else 0
    C=np.sum(w)-np.sum(w**2)/np.sum(w); tau2=max(0,(Q-df)/C) if C>0 else 0
    wr=1/(sd**2+tau2); bre=np.sum(wr*b)/np.sum(wr); sere=np.sqrt(1/np.sum(wr))
    return k,bfe,sefe,bre,sere,I2

print("\n=== SECTION-LEVEL FE meta (reproduce Fig7C) ===")
for p,g in assoc.groupby("program"):
    k,bfe,sefe,bre,sere,I2=meta(g.beta,g.se)
    print(f"{p:12s} n={k:2d} FE={bfe:+.4f} CI({bfe-1.96*sefe:+.4f},{bfe+1.96*sefe:+.4f}) p={2*norm.sf(abs(bfe/sefe)):.2e} I2={I2:4.0f}%")

# patient-level two-stage
def ivw(b,s): w=1/np.asarray(s,float)**2; return np.sum(w*b)/np.sum(w), np.sqrt(1/np.sum(w))
pat=[]
for (p,pt),g in assoc.groupby(["program","patient"]):
    b,s=ivw(g.beta.values,g.se.values); pat.append(dict(program=p,patient=pt,beta=b,se=s,nsec=len(g)))
pat=pd.DataFrame(pat)
print("\n=== PATIENT-LEVEL two-stage meta (8 patients): FE + RE + I2 ===")
for p,g in pat.groupby("program"):
    k,bfe,sefe,bre,sere,I2=meta(g.beta.values,g.se.values)
    print(f"{p:12s} n_pat={k} FE={bfe:+.4f} CI({bfe-1.96*sefe:+.4f},{bfe+1.96*sefe:+.4f}) pFE={2*norm.sf(abs(bfe/sefe)):.3e} | RE={bre:+.4f} CI({bre-1.96*sere:+.4f},{bre+1.96*sere:+.4f}) pRE={2*norm.sf(abs(bre/sere)):.3e} I2={I2:.0f}%")

---
### Reproducibility notes
- Key parameters are set inline in each section (Leiden resolution 0.5; DTP-enrichment threshold 0.6; DE log2FC>0.25 & BH padj<0.05;
  GSEA 1,000 permutations seed 42; Cox tertile High-vs-Low + age/lymph adjustment with Schoenfeld PH test; iLINCS max-z aggregation,
  GO-axis top-100 terms, >=1 target overlap, cross-lineage; spatial epithelial+depth-adjusted regression, inverse-variance meta-analysis).
- Software versions: see `requirements.txt`.
- Data accessions: GSE156246; Zenodo 4751624 (almaan/her2st); cBioPortal brca_metabric & TCGA; iLINCS.